# CIFAR-10 CNN benchmark: CUDA vs Apple Metal (MPS) vs CPU

This reproducible FP32 benchmark trains one CNN architecture on CIFAR-10 and measures its test accuracy, end-to-end training rate, model-only inference latency/throughput, and a synthetic training-step microbenchmark.

The backend is selected automatically: CUDA in Google Colab, MPS on supported Apple Silicon Macs, otherwise CPU. The CPU cells use the **same model and trained weights** as the selected accelerator.

## Run on this Mac

1. Select the `Python 3.12 (metal-benchmark)` kernel.
2. Run all cells. macOS uses `NUM_WORKERS = 0` deliberately: it is the reliable Jupyter/DataLoader setting.

## Run in Google Colab

1. Upload this notebook and choose **Runtime → Change runtime type → GPU**.
2. Run all cells. The notebook will select CUDA automatically.

The primary comparison is float32. CUDA AMP is intentionally excluded, and CUDA TF32 is disabled when the installed PyTorch supports that control. Compare runs with the same epochs, batch size, seed, and package versions.

In [ ]:
import json
import os
import platform
import random
import statistics
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("torchvision:", __import__("torchvision").__version__)
print("Platform:", platform.platform())
print("CUDA available:", torch.cuda.is_available())
print("MPS built:", torch.backends.mps.is_built())
print("MPS available:", torch.backends.mps.is_available())

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Selected device:", DEVICE)

if DEVICE.type == "cuda":
    print("CUDA GPU:", torch.cuda.get_device_name(0))
elif DEVICE.type == "mps":
    print("Apple Metal / MPS enabled")
elif platform.system() == "Darwin":
    if not torch.backends.mps.is_built():
        reason = "This PyTorch build does not include MPS support. Install an Apple-Silicon/macOS PyTorch wheel."
    else:
        reason = "MPS is built but unavailable: macOS does not expose a supported Metal device to this process."
    print("MPS diagnostic:", reason)
    print("CPU was selected only because MPS is unavailable; do not label this run as an MPS result.")

In [ ]:
# A successful operation is required before treating an MPS run as valid.
if DEVICE.type == "mps":
    try:
        mps_x = torch.randn(1024, 1024, dtype=torch.float32, device=DEVICE)
        mps_y = mps_x @ mps_x
        torch.mps.synchronize()
        print("MPS verification matrix-product mean:", mps_y.mean().item())
        del mps_x, mps_y
    except Exception as error:
        raise RuntimeError(
            "MPS was selected but a real Metal tensor operation failed. "
            "Do not continue with a CPU fallback; update macOS/PyTorch and rerun."
        ) from error

In [ ]:
SEED = 42
BATCH_SIZE = 128
EPOCHS = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0 if platform.system() == "Darwin" else 2
DATA_ROOT = Path("data")
DTYPE = torch.float32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # Keep the main CUDA workload FP32 rather than TF32/AMP where supported.
    try:
        torch.backends.cuda.matmul.fp32_precision = "ieee"
        torch.backends.cudnn.conv.fp32_precision = "ieee"
    except AttributeError:  # PyTorch versions before the fp32_precision API.
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False

print({
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "num_workers": NUM_WORKERS,
    "dtype": str(DTYPE),
})

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_dataset = datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=transform_test)

loader_options = {
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE.type == "cuda",  # pinning is useful only for CUDA transfers here
}
if NUM_WORKERS > 0:
    loader_options["persistent_workers"] = True

loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=loader_generator, **loader_options)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, **loader_options)

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))
print("Classes:", train_dataset.classes)

In [ ]:
class SmallCIFARNet(nn.Module):
    """Same FP32 CNN architecture on CUDA, MPS, and CPU."""

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        return self.classifier(torch.flatten(self.features(x), 1))


model = SmallCIFARNet().to(device=DEVICE, dtype=DTYPE)
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print("Trainable parameters:", f"{parameter_count:,}")

In [ ]:
def synchronize(device):
    """Wait for completed accelerator work so timings include executed kernels."""
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps":
        torch.mps.synchronize()


def move_batch(images, labels, device):
    # non_blocking needs pinned host memory; this notebook pins only CUDA loaders.
    return (
        images.to(device, dtype=DTYPE, non_blocking=device.type == "cuda"),
        labels.to(device, non_blocking=device.type == "cuda"),
    )


def accelerator_memory_mb(device):
    if device.type == "cuda":
        return {
            "allocated_mb": torch.cuda.memory_allocated(device) / 1024**2,
            "reserved_mb": torch.cuda.memory_reserved(device) / 1024**2,
            "peak_allocated_mb": torch.cuda.max_memory_allocated(device) / 1024**2,
            "peak_reserved_mb": torch.cuda.max_memory_reserved(device) / 1024**2,
        }
    if device.type == "mps":
        return {
            "allocated_mb": torch.mps.current_allocated_memory() / 1024**2,
            "driver_allocated_mb": torch.mps.driver_allocated_memory() / 1024**2,
        }
    return {}

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats(DEVICE)


def run_epoch(model, loader, optimizer, criterion, device, training):
    model.train(training)
    total_loss = torch.zeros((), device=device, dtype=DTYPE)
    total_correct = torch.zeros((), device=device, dtype=torch.int64)
    total_samples = 0

    synchronize(device)
    start = time.perf_counter()
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for images, labels in loader:
            images, labels = move_batch(images, labels, device)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                loss.backward()
                optimizer.step()
            batch_samples = labels.size(0)
            total_loss += loss.detach() * batch_samples
            total_correct += (logits.argmax(dim=1) == labels).sum()
            total_samples += batch_samples
    synchronize(device)
    elapsed = time.perf_counter() - start
    return {
        "loss": (total_loss / total_samples).item(),
        "accuracy": (total_correct.float() / total_samples).item(),
        "seconds": elapsed,
        "images_per_second": total_samples / elapsed,
    }


training_history = []
for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, optimizer, criterion, DEVICE, training=True)
    test_metrics = run_epoch(model, test_loader, optimizer, criterion, DEVICE, training=False)
    training_history.append({"epoch": epoch, "train": train_metrics, "test": test_metrics})
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {train_metrics['loss']:.4f} | train acc {100 * train_metrics['accuracy']:.2f}% | "
        f"test loss {test_metrics['loss']:.4f} | test acc {100 * test_metrics['accuracy']:.2f}% | "
        f"epoch time {train_metrics['seconds']:.2f}s | train {train_metrics['images_per_second']:.1f} img/s"
    )

In [ ]:
CHECKPOINT_PATH = Path("cifar10_cnn.pt")
torch.save({
    "model_state_dict": model.state_dict(),
    "parameter_count": parameter_count,
    "classes": train_dataset.classes,
    "training_history": training_history,
    "pytorch_version": torch.__version__,
    "dtype": "float32",
}, CHECKPOINT_PATH)
checkpoint_size_mb = CHECKPOINT_PATH.stat().st_size / 1024**2

print("Saved:", CHECKPOINT_PATH.resolve())
print("Checkpoint size:", f"{checkpoint_size_mb:.3f} MB")
print("Final test accuracy:", f"{100 * training_history[-1]['test']['accuracy']:.2f}%")
print("Accelerator memory:", accelerator_memory_mb(DEVICE))

## Model-only inference benchmark

Inputs are synthetic `[B, 3, 32, 32]` float32 tensors. Dataset loading, preprocessing, allocation, and host-to-device transfer are excluded. Each measured iteration synchronizes before and after the forward pass, so CUDA/MPS timing captures finished accelerator work rather than asynchronous dispatch.

In [ ]:
@torch.inference_mode()
def benchmark_inference(model, device, batch_sizes=(1, 8, 32, 128), warmup=30, iterations=200):
    model.eval()
    rows = []
    for batch_size in batch_sizes:
        inputs = torch.randn(batch_size, 3, 32, 32, dtype=DTYPE, device=device)
        for _ in range(warmup):
            model(inputs)
        synchronize(device)
        latency_ms = []
        for _ in range(iterations):
            synchronize(device)
            start = time.perf_counter()
            model(inputs)
            synchronize(device)
            latency_ms.append((time.perf_counter() - start) * 1_000)
        mean_ms = statistics.fmean(latency_ms)
        row = {
            "batch_size": batch_size,
            "mean_latency_ms": mean_ms,
            "p50_latency_ms": float(np.percentile(latency_ms, 50)),
            "p95_latency_ms": float(np.percentile(latency_ms, 95)),
            "throughput_images_per_second": batch_size / (mean_ms / 1_000),
        }
        rows.append(row)
        print(
            f"batch={batch_size:3d} | mean={row['mean_latency_ms']:8.3f} ms | "
            f"p50={row['p50_latency_ms']:8.3f} ms | p95={row['p95_latency_ms']:8.3f} ms | "
            f"throughput={row['throughput_images_per_second']:10.1f} img/s"
        )
    return rows


inference_results = benchmark_inference(model, DEVICE)

## CPU baseline

This copies the trained FP32 checkpoint to a CPU instance of the same architecture. It intentionally does not retrain, so inference is a direct same-weights comparison. The synthetic training-step benchmark below is also run on CPU.

In [ ]:
cpu_model = SmallCIFARNet().to(device="cpu", dtype=DTYPE).eval()
cpu_model.load_state_dict(model.state_dict())
cpu_inference_results = benchmark_inference(cpu_model, torch.device("cpu"), warmup=10, iterations=100)

## Synthetic training-step microbenchmark

This times forward pass, loss, backward pass, and `optimizer.step()` together, without DataLoader or augmentation time. Warm-up performs optimizer-state allocation and accelerator compilation before timing.

In [ ]:
def benchmark_training_step(device, batch_size=128, warmup=10, iterations=50):
    benchmark_model = SmallCIFARNet().to(device=device, dtype=DTYPE).train()
    benchmark_optimizer = torch.optim.AdamW(benchmark_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    benchmark_criterion = nn.CrossEntropyLoss()
    inputs = torch.randn(batch_size, 3, 32, 32, dtype=DTYPE, device=device)
    targets = torch.randint(0, 10, (batch_size,), device=device)

    def step():
        benchmark_optimizer.zero_grad(set_to_none=True)
        loss = benchmark_criterion(benchmark_model(inputs), targets)
        loss.backward()
        benchmark_optimizer.step()

    for _ in range(warmup):
        step()
    synchronize(device)
    times_ms = []
    for _ in range(iterations):
        synchronize(device)
        start = time.perf_counter()
        step()
        synchronize(device)
        times_ms.append((time.perf_counter() - start) * 1_000)
    mean_ms = statistics.fmean(times_ms)
    result = {
        "batch_size": batch_size,
        "mean_step_ms": mean_ms,
        "p50_step_ms": float(np.percentile(times_ms, 50)),
        "p95_step_ms": float(np.percentile(times_ms, 95)),
        "training_images_per_second": batch_size / (mean_ms / 1_000),
    }
    print(result)
    return result


training_step_accelerator = benchmark_training_step(DEVICE)
training_step_cpu = benchmark_training_step(torch.device("cpu"))

In [ ]:
def device_name(device):
    if device.type == "cuda":
        return torch.cuda.get_device_name(0)
    if device.type == "mps":
        return "Apple Metal / MPS"
    return platform.processor() or platform.machine()

results = {
    "system": {
        "device": str(DEVICE),
        "device_name": device_name(DEVICE),
        "platform": platform.platform(),
        "python_version": platform.python_version(),
        "pytorch_version": torch.__version__,
        "torchvision_version": __import__("torchvision").__version__,
        "cuda_version": torch.version.cuda,
        "mps_built": torch.backends.mps.is_built(),
        "mps_available": torch.backends.mps.is_available(),
    },
    "configuration": {
        "precision": "float32",
        "seed": SEED,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "num_workers": NUM_WORKERS,
    },
    "model": {"name": "SmallCIFARNet", "parameter_count": parameter_count, "checkpoint_size_mb": checkpoint_size_mb, "input_shape": [3, 32, 32]},
    "training": {"history": training_history, "training_step_accelerator": training_step_accelerator, "training_step_cpu": training_step_cpu},
    "test": {"final_loss": training_history[-1]["test"]["loss"], "final_accuracy": training_history[-1]["test"]["accuracy"]},
    "inference_accelerator": inference_results,
    "inference_cpu_same_machine": cpu_inference_results,
    "accelerator_memory_mb": accelerator_memory_mb(DEVICE),
}
RESULTS_PATH = Path(f"benchmark_{DEVICE.type}.json")
RESULTS_PATH.write_text(json.dumps(results, indent=2), encoding="utf-8")
print("Saved benchmark:", RESULTS_PATH.resolve())
print(json.dumps(results["system"], indent=2))

## Report checklist

Compare only matching FP32 runs. For each CUDA, MPS, and CPU result report the backend/device name, package versions, parameter count, final test accuracy, epoch images/s, training-step images/s, and inference mean/p50/p95 latency plus throughput for batch sizes 1, 8, 32, and 128.

Batch 1 is the most relevant interactive latency measurement; larger batches show throughput scaling. Do not compare a CUDA AMP/TF32 run with this FP32 MPS run as a pure hardware result.